# 03 — Direct Preference Optimization (DPO) on CPU

**DPO** learns from *preferences* instead of target answers: for each prompt, a preferred ("chosen") answer and a
worse ("rejected") one. It raises the likelihood of the chosen answer relative to the rejected one, compared to a
reference model. With a LoRA adapter the reference is simply the base model with the adapter switched off, so no
second model is loaded.

Same booking task as notebook 01. The chosen answer is the exact JSON; the rejected one is the kind of answer the
base model gives (JSON in a markdown code block, "sunday", "5:15pm") or a subtly wrong value.

Two jobs run in parallel to show a classic pitfall:

| job | loss | what to expect |
|---|---|---|
| `dpo` | DPO only (`sigmoid`) | the gap between chosen and rejected grows, but the model can widen it by moving away from *both*, e.g. by dropping JSON altogether |
| `dpo-sft` | DPO + an SFT term on the chosen answer (`sigmoid+sft`) | keeps the chosen answer likely while learning the preference |

Each job measures exact JSON match and *preference accuracy* (how often the model scores the chosen answer above
the rejected one) before and after 40 steps. Expected: ~6 minutes.

In [ ]:
%pip install -q "sagemaker==3.21.0"

In [ ]:
import sys

sys.path.insert(0, "scripts")
from sm_helpers import Lab

lab = Lab()
CAPACITY = "fallback"   # "on-demand" | "spot" | "fallback" (compared side by side in 01-sft.ipynb)

In [ ]:
import booking_data

train, _ = booking_data.make_splits()
pair = booking_data.dpo_pairs(train[:1])[0]
print(f"request:  {pair['request']}\nchosen:   {pair['chosen']}\nrejected: {pair['rejected']}")

## Train both variants

In [ ]:
jobs = {
    "dpo": lab.submit("dpo", "train_dpo.py", {"steps": 40, "loss_type": "sigmoid"},
                      capacity=CAPACITY, requirements="requirements-trl.txt"),
    "dpo-sft": lab.submit("dpo-sft", "train_dpo.py", {"steps": 40, "loss_type": "sigmoid+sft"},
                          capacity=CAPACITY, requirements="requirements-trl.txt"),
}

In [ ]:
jobs = globals().get("jobs") or {k: lab.latest(k) for k in ("dpo", "dpo-sft")}   # after a kernel restart
desc = lab.wait(*jobs.values())

## Results

In [ ]:
print(f"{'':16} {'exact JSON':>10} {'prefers chosen':>15}")
for name, job in jobs.items():
    m = lab.metrics(job)
    if name == "dpo":
        print(f"{'base':16} {m['base']['exact_match']:10.0%} {m['base']['preference_accuracy']:15.0%}")
    print(f"{name:16} {m['finetuned']['exact_match']:10.0%} {m['finetuned']['preference_accuracy']:15.0%}")
print()
for name, job in jobs.items():
    ex = lab.metrics(job)["examples"][0]
    print(f"{name:8} {ex['finetuned']!r}")
print(f"{'target':8} {ex['target']}")

Both variants learn the preference (they score the chosen answer higher), but only DPO + SFT also *produces* the
chosen answer. This is why preference tuning usually starts from an SFT model, or adds an SFT term as here.